In [ ]:
!gdown 14lX_JgofYZLbIkjPf7rXmBOpVIfMudDo -O kaggle.json
!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download thedevastator/dailydialog-multi-turn-dialog-with-intention-and
!unzip dailydialog-multi-turn-dialog-with-intention-and.zip -d .

Downloading...
From: https://drive.google.com/uc?id=14lX_JgofYZLbIkjPf7rXmBOpVIfMudDo
To: /content/kaggle.json
100% 69.0/69.0 [00:00<00:00, 282kB/s]
Dataset URL: https://www.kaggle.com/datasets/thedevastator/dailydialog-multi-turn-dialog-with-intention-and
License(s): CC0-1.0
Archive:  dailydialog-multi-turn-dialog-with-intention-and.zip
  inflating: ./test.csv              
  inflating: ./train.csv             
  inflating: ./validation.csv        


In [ ]:
!gdown 1G11jm_TDXVY4wAyogEG3qWE6H2q9d8mt -O results_zipped.zip
!unzip results_zipped.zip -d ./results


Downloading...
From (original): https://drive.google.com/uc?id=1G11jm_TDXVY4wAyogEG3qWE6H2q9d8mt
From (redirected): https://drive.google.com/uc?id=1G11jm_TDXVY4wAyogEG3qWE6H2q9d8mt&confirm=t&uuid=1861ff52-0607-48c4-b951-95439d767957
To: /content/results_zipped.zip
100% 424M/424M [00:07<00:00, 60.5MB/s]
Archive:  results_zipped.zip
   creating: ./results/checkpoint-1500/
  inflating: ./results/checkpoint-1500/config.json  
  inflating: ./results/checkpoint-1500/tokenizer_config.json  
  inflating: ./results/checkpoint-1500/training_args.bin  
  inflating: ./results/checkpoint-1500/trainer_state.json  
  inflating: ./results/checkpoint-1500/model.safetensors  
  inflating: ./results/checkpoint-1500/merges.txt  
  inflating: ./results/checkpoint-1500/optimizer.pt  
  inflating: ./results/checkpoint-1500/rng_state.pth  
  inflating: ./results/checkpoint-1500/scaler.pt  
  inflating: ./results/checkpoint-1500/scheduler.pt  
  inflating: ./results/checkpoint-1500/vocab.json  
  inflating: ./

In [ ]:
import pandas as pd
import ast
import re

# Function to fix act and emotion columns (adds missing commas and converts to list)
def fix_and_eval_list(s):
    if isinstance(s, list):
        return s
    s_fixed = re.sub(r"(\d)\s+(?=\d)", r"\1, ", s)
    return ast.literal_eval(s_fixed)

# Function to parse dialog and extract utterances (between quotes)
def parse_dialog(s):
    utterances = re.findall(r"'(.*?)'|\"(.*?)\"", s)
    return [u[0] if u[0] else u[1] for u in utterances]

def load_and_prepare_flat_df(path, tokenizer, max_length=64):
    # Load CSV
    df = pd.read_csv(path)

    # Parse columns
    df["dialog"] = df["dialog"].apply(parse_dialog)
    df["act"] = df["act"].apply(fix_and_eval_list)
    df["emotion"] = df["emotion"].apply(fix_and_eval_list)

    # Flatten
    rows = []
    for dialog_id, row in df.iterrows():
        for turn_id, (utt, emo) in enumerate(zip(row["dialog"], row["emotion"])):
            rows.append({
                "dialog_id": dialog_id,
                "turn_id": turn_id,
                "utterance": utt,
                "label": emo
            })
    df_flat = pd.DataFrame(rows)

    # Tokenize (lists of token IDs and masks)
    tokenized = tokenizer(
        df_flat["utterance"].tolist(),
        padding=True,
        truncation=True,
        max_length=max_length
    )
    df_flat["input_ids"] = tokenized["input_ids"]
    df_flat["attention_mask"] = tokenized["attention_mask"]

    return df_flat



In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch

# Load RoBERTa base tokenizer and model (frozen)
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')



full_model = RobertaForSequenceClassification.from_pretrained("./results/checkpoint-1500")
model = full_model.roberta  # Backbone RoBERTa fine-tunato
model.eval()  # Evaluation mode (disable dropout)

# If using GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [ ]:
df_train_flat = load_and_prepare_flat_df("train.csv", tokenizer)
df_val_flat = load_and_prepare_flat_df("validation.csv", tokenizer)
df_test_flat = load_and_prepare_flat_df("test.csv", tokenizer)

In [ ]:
df_train_flat.head()

,dialog_id,turn_id,utterance,label,input_ids,attention_mask
0,0,0,"Say , Jim , how about going for a few beers af...",0,"[0, 34673, 2156, 2488, 2156, 141, 59, 164, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,0,1,You know that is tempting but is really not g...,0,"[0, 370, 216, 14, 16, 25057, 53, 16, 269, 45, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,0,2,What do you mean ? It will help us to relax .,0,"[0, 653, 109, 47, 1266, 17487, 85, 40, 244, 20...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,0,3,Do you really think so ? I don't . It will ju...,0,"[0, 1832, 47, 269, 206, 98, 17487, 38, 218, 75...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,0,4,I guess you are right.But what shall we do ? ...,0,"[0, 38, 4443, 47, 32, 235, 4, 1708, 99, 5658, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
import torch
from tqdm import tqdm
import numpy as np

# Store embeddings and labels
embeddings = []
labels = []

# Loop over the dataframe
for idx, row in tqdm(df_train_flat.iterrows(), total=len(df_train_flat)):
    input_ids = torch.tensor([row["input_ids"]]).to(device)
    attention_mask = torch.tensor([row["attention_mask"]]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # [1, seq_len, hidden_dim]

        # mean pooling: sum(hidden * mask) / sum(mask)
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()  # [1, seq_len, hidden_dim]
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)  # [1, hidden_dim]
        counts = mask.sum(dim=1)  # [1, hidden_dim] (broadcasted)
        mean_pooled = summed / counts.clamp(min=1e-9)  # avoid division by zero

        embeddings.append(mean_pooled.cpu().numpy().squeeze())
        labels.append(row["label"])

# Convert to numpy arrays
embeddings = np.array(embeddings)
labels = np.array(labels)

# Save to disk
np.save("train_embeddings.npy", embeddings)
np.save("train_labels.npy", labels)


100%|██████████| 87170/87170 [13:08<00:00, 110.58it/s]


In [ ]:
import torch
from tqdm import tqdm
import numpy as np

# Store embeddings and labels
val_embeddings = []
val_labels = []

# Loop over the dataframe
for idx, row in tqdm(df_val_flat.iterrows(), total=len(df_val_flat)):
    input_ids = torch.tensor([row["input_ids"]]).to(device)
    attention_mask = torch.tensor([row["attention_mask"]]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # [1, seq_len, hidden_dim]

        # mean pooling con attention mask
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts.clamp(min=1e-9)

        val_embeddings.append(mean_pooled.cpu().numpy().squeeze())
        val_labels.append(row["label"])

# Convert to numpy arrays
val_embeddings = np.array(val_embeddings)
val_labels = np.array(val_labels)

# Save to disk
np.save("val_embeddings.npy", val_embeddings)
np.save("val_labels.npy", val_labels)


100%|██████████| 8069/8069 [01:12<00:00, 111.31it/s]


In [ ]:
import torch
from tqdm import tqdm
import numpy as np

# Store embeddings and labels
test_embeddings = []
test_labels = []

# Loop over the dataframe
for idx, row in tqdm(df_test_flat.iterrows(), total=len(df_test_flat)):
    input_ids = torch.tensor([row["input_ids"]]).to(device)
    attention_mask = torch.tensor([row["attention_mask"]]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # [1, seq_len, hidden_dim]

        # mean pooling con attention mask
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts.clamp(min=1e-9)

        test_embeddings.append(mean_pooled.cpu().numpy().squeeze())
        test_labels.append(row["label"])

# Convert to numpy arrays
test_embeddings = np.array(test_embeddings)
test_labels = np.array(test_labels)

# Save to disk
np.save("test_embeddings.npy", test_embeddings)
np.save("test_labels.npy", test_labels)


100%|██████████| 7740/7740 [01:08<00:00, 112.44it/s]


In [ ]:
import zipfile

# Train set
with zipfile.ZipFile('train_embeddings_labels_fine_tuned.zip', 'w') as zipf:
    zipf.write('train_embeddings.npy')
    zipf.write('train_labels.npy')

# Validation set
with zipfile.ZipFile('val_embeddings_labels_fine_tuned.zip', 'w') as zipf:
    zipf.write('val_embeddings.npy')
    zipf.write('val_labels.npy')

# Test set
with zipfile.ZipFile('test_embeddings_labels_fine_tuned.zip', 'w') as zipf:
    zipf.write('test_embeddings.npy')
    zipf.write('test_labels.npy')
